# Proyecto - Asistente de IA para aerolíneas

Ahora, reuniremos lo que hemos aprendido para crear un asistente de IA para atención al cliente para una aerolínea.

In [8]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [9]:
# Inicialización

load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key Sin Configurar")
    
MODEL = "gpt-4o-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [10]:
system_message = "Eres un asistente útil para una aerolínea llamada FlightAI. "
system_message += "Da respuestas breves y corteses, de no más de una oración. "
system_message += "Se siempre preciso. Si no sabes la respuesta, dilo."

In [4]:
# Esta función parece bastante más simple que la de mi video, porque estamos aprovechando las últimas actualizaciones de Gradio.

def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7875

To create a public link, set `share=True` in `launch()`.


## Herramientas

Las herramientas son una característica increíblemente poderosa que ofrecen los LLM de vanguardia.

Con las herramientas, puedes escribir una función y hacer que el LLM llame a esa función como parte de su respuesta.

Suena casi espeluznante... ¿le estamos dando el poder de ejecutar código en nuestra máquina?

Bueno, más o menos.

In [11]:
# Vamos a hacer una función útil

ticket_prices = {"londres": "799€", "parís": "899€", "tokyo": "1400€", "berlín": "499€"}

def get_ticket_price(destination_city):
    print(f"Se solicitó la herramienta get_ticket_price para {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")

In [12]:
get_ticket_price("Berlín")

Se solicitó la herramienta get_ticket_price para Berlín


'499€'

In [13]:
# Hay una estructura de diccionario particular que se requiere para describir nuestra función:

price_function = {
    "name": "get_ticket_price",
    "description": "Obtén el precio de un billete de ida y vuelta a la ciudad de destino. Llámalo siempre que necesites saber el precio del billete, por ejemplo, cuando un cliente pregunte '¿Cuánto cuesta un billete a esta ciudad?'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "La ciudad a la que el cliente desea viajar",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [14]:
# Y esto está incluido en una lista de herramientas:

tools = [{"type": "function", "function": price_function}]

## Cómo hacer que OpenAI use nuestra herramienta

Hay algunos detalles complicados para permitir que OpenAI "llame a nuestra herramienta"

Lo que realmente hacemos es darle al LLM la oportunidad de informarnos que quiere que ejecutemos la herramienta.

Así es como se ve la nueva función de chat:

In [40]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    print(f'Inicio: {response}', '\n')

    if response.choices[0].finish_reason=="tool_calls":
        print('0. razón tool_calls')
        message = response.choices[0].message
        print(f'1. mensaje: {message}')
        response, city = handle_tool_call(message)
        print('6.', response, city)
        messages.append(message)
        messages.append(response)
        print(f'7. Mensajes finales: {messages}')
        response = openai.chat.completions.create(model=MODEL, messages=messages)

    else:
        print('no se usa tool')
    
    return messages, history, response, response.choices[0].message.content

In [41]:
#Tenemos que escribir esa función handle_tool_call:

def handle_tool_call(message):
    print('2. entrando en handle')
    tool_call = message.tool_calls[0]
    print('3.', tool_call)
    arguments = json.loads(tool_call.function.arguments)
    print('4.', arguments)
    city = arguments.get('destination_city')
    price = get_ticket_price(city)
    print('5.', price)
    response = {
        "role": "tool",
        "content": json.dumps({"destination_city": city,"price": price}),
        "tool_call_id": message.tool_calls[0].id
    }
    return response, city

In [42]:
messages, history, response, mensaje = chat(message='Cuánto cuesta un vuelo a Tokyo?'
     , history=[])

Inicio: ChatCompletion(id='chatcmpl-ECos56N8CBlhl4rBuMWJ6PHgKCAcN', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_kjydvfr0gADdM9Izrkr2Lpmg', function=Function(arguments='{"destination_city":"Tokyo"}', name='get_ticket_price'), type='function')]))], created=1786724193, model='gpt-4o-mini-2024-07-18', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_d6d3876e6e', usage=CompletionUsage(completion_tokens=16, prompt_tokens=153, total_tokens=169, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0))) 

0. razón tool_calls
1. mensaje: ChatCompletionMessage(conte

In [43]:
mensaje

'El vuelo a Tokyo cuesta aproximadamente 1400€.'

In [48]:
for i in range(len(messages)):
    print(i)
    print(messages[i])
    print('\n')

0
{'role': 'system', 'content': 'Eres un asistente útil para una aerolínea llamada FlightAI. Da respuestas breves y corteses, de no más de una oración. Se siempre preciso. Si no sabes la respuesta, dilo.'}


1
{'role': 'user', 'content': 'Cuánto cuesta un vuelo a Tokyo?'}


2
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_kjydvfr0gADdM9Izrkr2Lpmg', function=Function(arguments='{"destination_city":"Tokyo"}', name='get_ticket_price'), type='function')])


3
{'role': 'tool', 'content': '{"destination_city": "Tokyo", "price": "1400\\u20ac"}', 'tool_call_id': 'call_kjydvfr0gADdM9Izrkr2Lpmg'}




In [49]:
history

[]

In [50]:
response

ChatCompletion(id='chatcmpl-ECos5kQVUhbFveoNVZspbWfXvZAjP', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='El vuelo a Tokyo cuesta aproximadamente 1400€.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1786724193, model='gpt-4o-mini-2024-07-18', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_6339e52f56', usage=CompletionUsage(completion_tokens=10, prompt_tokens=103, total_tokens=113, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0)))

In [14]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7877

To create a public link, set `share=True` in `launch()`.


Se solicitó la herramienta get_ticket_price para Londres


# Anexo: cómo se declaran las herramientas en 2026

El `price_function` de arriba es el formato de **Chat Completions**, donde la función va envuelta
dentro de una llave `"function"`. Sigue funcionando, pero dos cosas envejecieron:

1. **El diccionario ya casi no se escribe a mano.** Los cuatro SDKs derivan el esquema de la firma
   y el docstring de la función de Python. El diccionario existe como formato de cable, no como algo
   que tú redactes.
2. **El `chat()` de arriba atiende una sola llamada** (`message.tool_calls[0]`) y no itera. Los
   modelos actuales emiten varias llamadas en paralelo en una misma respuesta y encadenan varias
   rondas, así que ese `[0]` descarta llamadas en silencio.

Abajo está la misma herramienta `get_ticket_price` implementada en los cuatro frameworks. Todos
resuelven el mismo prompt con **dos ciudades a la vez**, para que se vea el manejo en paralelo.

In [ ]:
# Herramienta compartida por los ejemplos de abajo.
# Dos diferencias con la versión de arriba:
#   1. Lleva anotaciones de tipo y docstring con sección Args: de ahí es de donde los SDKs
#      modernos generan el esquema JSON solos.
#   2. Normaliza las tildes. Las claves de ticket_prices son "berlín" y "parís", pero el modelo
#      a veces escribe "Berlin" sin tilde y la búsqueda fallaba de forma intermitente.
import unicodedata


def _sin_tildes(texto: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", texto.lower())
                   if unicodedata.category(c) != "Mn")


PRECIOS = {_sin_tildes(ciudad): precio for ciudad, precio in ticket_prices.items()}


def get_ticket_price(destination_city: str) -> str:
    """Obtén el precio de un billete de ida y vuelta a la ciudad de destino.

    Args:
        destination_city: La ciudad a la que el cliente desea viajar.
    """
    print(f"  [herramienta ejecutada] get_ticket_price({destination_city})")
    return PRECIOS.get(_sin_tildes(destination_city), "Desconocido")


# Prompt de prueba: pide dos ciudades a la vez para forzar llamadas en paralelo
PREGUNTA = "¿Cuánto cuesta ir a Berlín y a Tokyo?"

In [62]:
tools_openai = [{
    "type": "function",
    "name": "get_ticket_price",
    "description": "Obtén el precio de un billete de ida y vuelta a la ciudad de destino.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {"type": "string", "description": "La ciudad a la que el cliente desea viajar"},
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    },
    "strict": True,
}]


def chat_openai(pregunta):
    entrada = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": pregunta},
    ]
    while True:
        respuesta = openai.responses.create(model=MODEL, input=entrada, tools=tools_openai)
        print('\n', respuesta, '\n')
        llamadas = [item for item in respuesta.output if item.type == "function_call"]
        # print(llamadas[0])
        if not llamadas:                      # ya no quiere herramientas: terminó
            print('entrada final')
            print(entrada)
            return respuesta.output_text
        entrada += respuesta.output           # se conserva el turno del asistente completo
        for llamada in llamadas:              # se atienden TODAS, no solo la primera
            argumentos = json.loads(llamada.arguments)
            entrada.append({
                "type": "function_call_output",
                "call_id": llamada.call_id,
                "output": get_ticket_price(argumentos["destination_city"]),
            })

    


print(chat_openai(PREGUNTA))


 Response(id='resp_0362accb1f8e93d4006a7f4574c5dc81a0ac916b635cc26cee', created_at=1786725748.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFunctionToolCall(arguments='{"destination_city":"Berlín"}', call_id='call_tiVNxZD0tQsf4LaUOU3em58e', name='get_ticket_price', type='function_call', id='fc_0362accb1f8e93d4006a7f4576811081a0bec89f826f8dd745', caller=None, namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"destination_city":"Tokyo"}', call_id='call_TOlFpSVLV0pNGtGuaLzHosDP', name='get_ticket_price', type='function_call', id='fc_0362accb1f8e93d4006a7f4576812481a0ae1395e13f44c340', caller=None, namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='get_ticket_price', parameters={'type': 'object', 'properties': {'destination_city': {'type': 'string', 'description': 'La ciudad a la que el cliente dese

In [ ]:
import anthropic
from anthropic import beta_tool

claude = anthropic.Anthropic()

@beta_tool
def get_ticket_price_claude(destination_city: str) -> str:
    """Obtén el precio de un billete de ida y vuelta a la ciudad de destino.

    Args:
        destination_city: La ciudad a la que el cliente desea viajar.
    """
    return get_ticket_price(destination_city)


# El decorador ya construyó el esquema: esto es lo que viaja a la API
print(json.dumps(get_ticket_price_claude.to_dict(), indent=2, ensure_ascii=False))

{
  "name": "get_ticket_price_claude",
  "description": "Obtén el precio de un billete de ida y vuelta a la ciudad de destino.",
  "input_schema": {
    "additionalProperties": false,
    "properties": {
      "destination_city": {
        "description": "La ciudad a la que el cliente desea viajar.",
        "title": "Destination City",
        "type": "string"
      }
    },
    "required": [
      "destination_city"
    ],
    "type": "object"
  }
}


Y el ciclo completo, sin escribir un solo `while`:

In [54]:
def chat_claude(pregunta):
    # tool_runner devuelve un iterador: cada vuelta es un turno del modelo.
    # El SDK ejecuta la función y reenvía el resultado por su cuenta; no hay bucle que escribir.
    runner = claude.beta.messages.tool_runner(
        model="claude-opus-5",
        max_tokens=1000,
        system=system_message,
        tools=[get_ticket_price_claude],
        messages=[{"role": "user", "content": pregunta}],
    )
    mensaje_final = None
    for mensaje in runner:
        mensaje_final = mensaje
    return "".join(b.text for b in mensaje_final.content if b.type == "text")


print(chat_claude(PREGUNTA))

  [herramienta ejecutada] get_ticket_price(Berlín)
  [herramienta ejecutada] get_ticket_price(Tokyo)
El billete de ida y vuelta a Berlín cuesta 499 € y a Tokyo 1400 €.


In [55]:
from google import genai
from google.genai import types

gemini = genai.Client(
    vertexai=True,
    project=os.getenv("GOOGLE_CLOUD_PROJECT"),
    location=os.getenv("GOOGLE_CLOUD_REGION", "us-east1"),
)


def chat_gemini(pregunta):
    respuesta = gemini.models.generate_content(
        model="gemini-2.5-flash",
        contents=pregunta,
        config=types.GenerateContentConfig(
            system_instruction=system_message,
            tools=[get_ticket_price],          # la función, no un diccionario
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ),
    )
    # El historial deja ver los turnos que el SDK gestionó por su cuenta
    print(f"  (turnos resueltos automáticamente: {len(respuesta.automatic_function_calling_history or [])})")
    return respuesta.text


print(chat_gemini(PREGUNTA))

  [herramienta ejecutada] get_ticket_price(Berlín)
  [herramienta ejecutada] get_ticket_price(Tokyo)
  (turnos resueltos automáticamente: 3)
El precio del billete de ida y vuelta a Berlín es de 499€ y a Tokyo es de 1400€.


In [56]:
from langchain_core.tools import tool
from langchain.agents import create_agent


@tool
def get_ticket_price_lc(destination_city: str) -> str:
    """Obtén el precio de un billete de ida y vuelta a la ciudad de destino."""
    return get_ticket_price(destination_city)


# El decorador ya derivó todo esto de la función
print("nombre:      ", get_ticket_price_lc.name)
print("descripción: ", get_ticket_price_lc.description)
print("esquema:     ", json.dumps(get_ticket_price_lc.args_schema.model_json_schema(), ensure_ascii=False))

agente = create_agent(
    model=f"openai:{MODEL}",
    tools=[get_ticket_price_lc],
    system_prompt=system_message,
)

resultado = agente.invoke({"messages": [{"role": "user", "content": PREGUNTA}]})
print("\n" + resultado["messages"][-1].content)

nombre:       get_ticket_price_lc
descripción:  Obtén el precio de un billete de ida y vuelta a la ciudad de destino.
esquema:      {"description": "Obtén el precio de un billete de ida y vuelta a la ciudad de destino.", "properties": {"destination_city": {"title": "Destination City", "type": "string"}}, "required": ["destination_city"], "title": "get_ticket_price_lc", "type": "object"}
  [herramienta ejecutada] get_ticket_price(Berlín)
  [herramienta ejecutada] get_ticket_price(Tokyo)

El billete de ida y vuelta a Berlín cuesta 499€ y a Tokio 1400€.


## 5. Aclaración: OpenAI no es "más verboso", es que comparamos capas distintas

Las secciones 1 a 4 no son una comparación pareja: la 1 usa la **API cruda** de OpenAI y las otras
tres usan **SDKs de alto nivel**. OpenAI también tiene su capa de agentes, el *Agents SDK*, donde el
decorador `@function_tool` hace exactamente lo mismo que `@beta_tool` y `@tool` — saca el nombre de
la función, la descripción del docstring y el esquema de los type hints (usa `inspect` + `griffe` +
`pydantic` por debajo).

```bash
pip install openai-agents
```

Comparando capa contra capa, los cuatro quedan prácticamente idénticos.

In [ ]:
from agents import Agent, Runner, function_tool


@function_tool
def get_ticket_price_agents(destination_city: str) -> str:
    """Obtén el precio de un billete de ida y vuelta a la ciudad de destino.

    Args:
        destination_city: La ciudad a la que el cliente desea viajar.
    """
    return get_ticket_price(destination_city)


# Mismo patrón que Claude y LangChain: el decorador ya derivó todo
print("nombre:      ", get_ticket_price_agents.name)
print("descripción: ", get_ticket_price_agents.description)
print("esquema:     ", json.dumps(get_ticket_price_agents.params_json_schema, ensure_ascii=False))

agente_openai = Agent(
    name="FlightAI",
    model=MODEL,
    instructions=system_message,
    tools=[get_ticket_price_agents],
)

# Runner.run es asíncrono; en Jupyter se usa await directamente.
# En un script suelto sería asyncio.run(...) o Runner.run_sync(...).
resultado = await Runner.run(agente_openai, PREGUNTA)
print("\n" + resultado.final_output)

In [ ]:
from langgraph.prebuilt import ToolNode


@tool
def consultar_clima(city: str) -> str:
    """Consulta el clima de la ciudad."""
    return "soleado"


@tool
def reservar_vuelo(city: str) -> str:
    """Reserva un vuelo a la ciudad."""
    return "reserva confirmada"


agente_3 = create_agent(
    model=f"openai:{MODEL}",
    tools=[get_ticket_price_lc, consultar_clima, reservar_vuelo],   # TRES herramientas
)

print("NODOS del grafo:", list(agente_3.get_graph().nodes))
print("ARISTAS:        ", [(e.source, e.target) for e in agente_3.get_graph().edges])

# El nodo 'tools' es un ToolNode: un despachador por nombre, no una herramienta
nodo = ToolNode([get_ticket_price_lc, consultar_clima, reservar_vuelo])
print("\nUn solo ToolNode despacha:", list(nodo.tools_by_name.keys()))

Y ni siquiera hace falta un grafo. `bind_tools` solo adjunta los esquemas a la llamada del modelo y
te devuelve los `tool_calls` para que los ejecutes tú — es el equivalente de LangChain a lo que hace
la sección 1 a mano. El grafo aparece cuando quieres el ciclo automático:

In [ ]:
from langchain.chat_models import init_chat_model

modelo = init_chat_model(f"openai:{MODEL}").bind_tools([get_ticket_price_lc, consultar_clima])
respuesta = modelo.invoke(PREGUNTA)

# No se ejecutó nada: solo devuelve la intención del modelo
print("tool_calls:", respuesta.tool_calls)

## 6. Aclaración: en LangGraph la herramienta **no** es un nodo

Es una confusión común y cambia cómo diseñas el grafo. La herramienta es una función; el nodo es el
**ejecutor** de herramientas (`ToolNode`), y hay **uno solo** sin importar cuántas herramientas
registres. Ese nodo recibe los `tool_calls` del `AIMessage`, busca cada uno por nombre en su
despachador interno, los ejecuta —en paralelo si vienen varios— y devuelve los `ToolMessage`.

La celda de abajo lo comprueba: registra **tres** herramientas y muestra los nodos del grafo.

## Resumen

Comparando **capa contra capa**, que es la comparación justa:

| | API cruda | SDK de agentes | ¿Quién corre el ciclo? |
|---|---|---|---|
| **OpenAI** | Diccionario plano + `strict` (§1) | `@function_tool` (§5) | Tú / el `Runner` |
| **Claude** | `input_schema`, sin envoltura `"function"` | `@beta_tool` | Tú / el `tool_runner` |
| **Gemini** | `types.FunctionDeclaration` | La función tal cual en `tools=[...]` | El SDK (*automatic function calling*) |
| **LangChain** | `bind_tools` (§6) | `@tool` + `create_agent` | Tú / el grafo |

Dos conclusiones:

1. **La verbosidad es de la capa, no del proveedor.** OpenAI parecía el más extenso solo porque la
   §1 lo compara a nivel de API cruda contra SDKs de alto nivel. Con `@function_tool` los cuatro
   quedan iguales. Gemini es el único cuyo automatismo vive en el SDK base y no en una capa aparte.
2. **En LangGraph la herramienta no es un nodo.** Registres tres o veinte, sigue habiendo un solo
   nodo `tools` (un `ToolNode`) que despacha por nombre.

Un detalle que vale para producción: el ciclo automático **ejecuta tu función sin preguntar**. Para
herramientas con efectos secundarios — mover plata, escribir en una base, mandar correos — hay que
poner una compuerta de aprobación dentro de la función o interceptar la llamada antes de ejecutarla,
en vez de confiar en el runner. Este `get_ticket_price` solo lee un diccionario, así que no aplica.

Ahí es donde sí partirías el grafo en varios nodos en LangGraph: separar herramientas de solo
lectura, que pasan directo, de las que escriben, que requieren aprobación humana. Pero la división
es por política de riesgo, no una herramienta por nodo.

## 4. LangChain / LangGraph — `@tool` + `create_agent`

El decorador `@tool` genera nombre, descripción y esquema desde la función. El runtime cambió:
`create_react_agent` de LangGraph quedó **deprecado** a favor de `create_agent`, que vive en
`langchain.agents` y trae un sistema de middleware.

Nota de dependencias: el proveedor va en el string del modelo (`"openai:gpt-4o-mini"`), y cada uno
necesita su paquete. En este entorno está `langchain-openai`; para usar `"anthropic:claude-opus-5"`
o `"google_genai:..."` habría que instalar `langchain-anthropic` o `langchain-google-genai`.

## 3. Gemini — le pasas la función de Python directamente

El caso más extremo: **cero diccionarios y cero bucle**. Se pasa la función tal cual en `tools=[...]`
y el SDK genera el esquema desde el docstring, ejecuta la herramienta y continúa la conversación solo.
Eso se llama *Automatic Function Calling*.

Aquí se usa el SDK `google-genai` contra Vertex AI, o sea con las credenciales de la máquina y sin
`GOOGLE_API_KEY` (igual que en el día 2). Si quisieras el control manual, se desactiva con
`automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)` y ahí sí te toca leer
`response.function_calls` y responder tú.

## 2. Claude — decorador `@beta_tool` + `tool_runner`

Anthropic nunca tuvo la envoltura `"function"`: usa `input_schema` en vez de `parameters`. Pero lo
importante es que ya no escribes el diccionario ni el bucle:

- `@beta_tool` genera el esquema desde la firma y el docstring.
- `tool_runner` corre el ciclo completo — llama al modelo, ejecuta tu función, devuelve el resultado
  y repite hasta que Claude deja de pedir herramientas.

Requiere `ANTHROPIC_API_KEY` en el `.env`.

## 1. OpenAI — Responses API

El diccionario se aplanó: `type`, `name`, `description` y `parameters` van al mismo nivel, sin la
envoltura `"function"` de Chat Completions. Novedades frente al notebook:

- `strict: True` garantiza que los argumentos cumplan el esquema (requiere `additionalProperties: False`).
- Los resultados ya no van como un mensaje `role: "tool"`, sino como un ítem `function_call_output`
  que referencia el `call_id`.
- El bucle recorre **todas** las llamadas de la respuesta, no solo la primera.